# DEFINIÇÃO DO PROJETO

In [1]:
# Projeto do TechChallenger da Fase 3

# Este projeto apresenta o desenvolvimento de um Assistente Médico Inteligente utilizando
# modelos de linguagem ajustados com dados sintéticos.
# pipelines automatizados e ferramentas modernas de IA generativa.
# visando criar um sistema capaz de auxiliar o Médico no atendimento de emergência do hospital.
# suporte à decisão e consulta de protocolos internos.

# Arquitetura

# Consulta clínica contextualizada durante o atendimento - LangChain + Fine-Tuning.
# Orquestração de fluxos com LangGraph e pipeline de processamento com LangChain.
# Classificação automática de intenções do médico - LangGraph Router.
# Análise de sintomas com busca nos protocolos do hospital - RAG Semântico.
# Análise de exames laboratoriais com escala - CTCAE  GPT-4o + CTCAE.
# Verificação de exames e emissão de alertas - LangGraph + Notificações.
# Rastreabilidade e auditoria de todas as interações - Logs + Segurança.


# INSTALAÇÃO DAS BIBLIOTECAS E SETUP DO AMBIENTE

### Importação e Instalação das bibliotecas

In [2]:
# ========== INSTALAÇÃO DAS BIBLIOTECAS ==========
!pip install -q transformers datasets peft accelerate bitsandbytes
!pip install -q sentence-transformers faiss-cpu
!pip install -q pandas torch


In [3]:
# ========== IMPORTAÇÃO E INSTALAÇÃO DAS BIBLIOTECAS LANGGRAPH E LANGCHAIN ==========

import langgraph
import langchain
import importlib.metadata # Importa o módulo para obter metadados do pacote

# Consolidando todas as instalações pip no início
!pip install -U langgraph langchain langchain-text-splitters langchain-community faiss-cpu langchain-huggingface


from langchain_text_splitters import RecursiveCharacterTextSplitter

# Versões do langchain e langgraph
print(f"LangChain version: {langchain.__version__}")

# Obtém a versão de langgraph usando importlib.metadata
try:
    langgraph_version = importlib.metadata.version('langgraph')
    print(f"LangGraph version: {langgraph_version}")
except importlib.metadata.PackageNotFoundError:
    print("LangGraph not found or version not available.")

LangChain version: 1.3.1
LangGraph version: 1.2.1


### Montagem do Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
import json
import re
import csv
from datetime import datetime
from pathlib import Path

# LangChain / RAG / Fine-tuning
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

# Hugging Face (fine-tuning)
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Caminho Google Drive para modelos e vectorstore
GOOGLE_DRIVE_PATH_FINE_TUNED_MODEL = Path("/content/drive/MyDrive/tech_challenge_3/fine-tuning")

# Caminho Google Drive para os dados sintéticos
SYNTHETIC_DATA_DRIVE_PATH = Path("/content/drive/MyDrive/tech_challenge_3/dados_sinteticos")

# Caminho Google Drive para os dados de log
LOGGING_DATA_DRIVE_PATH = Path("/content/drive/MyDrive/tech_challenge_3/logging")

# Diretórios para dados e modelos
DADOS_DIR = Path("/content/drive/MyDrive/tech_challenge_3/dados_sinteticos_processados")

MODELO_DIR = GOOGLE_DRIVE_PATH_FINE_TUNED_MODEL # Modelo fine-tuned será salvo/carregado do Drive
VECTORSTORE_PATH = GOOGLE_DRIVE_PATH_FINE_TUNED_MODEL / "faiss_index" # Vectorstore será salvo/carregado do Google Drive
LOG_OPERACOES = LOGGING_DATA_DRIVE_PATH / "logging-operações.csv"
LOG_CONSULTAS = LOGGING_DATA_DRIVE_PATH / "logging-consultas.csv"

# Cria os diretórios necessários no Google Drive
for d in [DADOS_DIR, MODELO_DIR, VECTORSTORE_PATH, SYNTHETIC_DATA_DRIVE_PATH, LOGGING_DATA_DRIVE_PATH, GOOGLE_DRIVE_PATH_FINE_TUNED_MODEL ]:
    d.mkdir(parents=True, exist_ok=True)
print("Imports e configuração OK.")

/tmp/ipykernel_2131/1528396262.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Imports e configuração OK.


# PIPELINE DE DADOS

## Etapa 1 – Produção e carregamento de dados sintéticos

Os arquivos com prefixo `dados-sinteticos-` já foram criados no projeto. Esta célula carrega e expõe os dados para as etapas seguintes.

In [6]:
def definir_caminho_valido(base_path_fallback=None):
    """
    Define qual caminho utilizar: Prioriza o Google Drive (para dados sintéticos),
    caso contrário usa o fallback (local).
    """
    # Usar SYNTHETIC_DATA_DRIVE_PATH para dados sintéticos
    if 'SYNTHETIC_DATA_DRIVE_PATH' in globals() and SYNTHETIC_DATA_DRIVE_PATH.exists():
        print(f"--- Utilizando Google Drive (dados sintéticos): {SYNTHETIC_DATA_DRIVE_PATH} ---")
        return SYNTHETIC_DATA_DRIVE_PATH

    fallback = Path(base_path_fallback) if base_path_fallback else Path(".")
    print(f"--- Google Drive para dados sintéticos não encontrado. Utilizando local: {fallback.absolute()} ---")
    return fallback

def listar_arquivos_dados_sinteticos(caminho_alvo):
    """Lista arquivos com prefixo 'dataset-' no diretório escolhido."""
    if not caminho_alvo.exists():
        return []

    arquivos = list(caminho_alvo.glob("dataset-*"))
    return [str(f) for f in arquivos]

def carregar_json(path):
    """Carrega um arquivo JSON."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def carregar_todos_dados_sinteticos(lista_arquivos):
    """Carrega os arquivos JSON da lista fornecida em um dicionário."""
    dados = {}
    if not lista_arquivos:
        return dados

    for path in lista_arquivos:
        nome = Path(path).stem
        if path.endswith(".json"):
            try:
                dados[nome] = carregar_json(path)
            except Exception as e:
                dados[nome] = {"erro": str(e)}
    return dados

# 1. Definir o diretório de trabalho (Colab vs Local)
if os.path.exists("/content"):
    os.chdir("/content")
    base_local = Path("/content")
else:
    base_local = Path(".")

dados_brutos = {}

# 2. Identifica qual caminho está disponível (Drive ou Local) para dados sintéticos
diretorio_trabalho = definir_caminho_valido(base_local)

# 3. Lista os arquivos no diretório escolhido
arquivos_encontrados = listar_arquivos_dados_sinteticos(diretorio_trabalho)

if arquivos_encontrados:
    print("Arquivos encontrados:", [Path(f).name for f in arquivos_encontrados])
    # 4. Carrega os dados
    dados_brutos = carregar_todos_dados_sinteticos(arquivos_encontrados)

    for k, v in dados_brutos.items():
        print(f"  {k}: {len(v) if isinstance(v, list) else 'dict'} itens")
else:
    print("Nenhum arquivo 'dados-sinteticos-*' encontrado em nenhum dos locais.")

--- Utilizando Google Drive (dados sintéticos): /content/drive/MyDrive/tech_challenge_3/dados_sinteticos ---
Arquivos encontrados: ['dataset-pedidos-medicos.json', 'dataset-perguntas-respostas-medicos.json', 'dataset-pacientes.json', 'dataset-faq-medica.json', 'dataset-receitas.json', 'dataset-laudos.json', 'dataset-procedimentos.json', 'dataset-protocolos-medicos.json']
  dataset-pedidos-medicos: 7 itens
  dataset-perguntas-respostas-medicos: 7 itens
  dataset-pacientes: 10 itens
  dataset-faq-medica: 10 itens
  dataset-receitas: dict itens
  dataset-laudos: 3 itens
  dataset-procedimentos: dict itens
  dataset-protocolos-medicos: 8 itens


## Etapa 2 – Pipeline de anonimização

Substitui nomes de pessoas e dados de identificação (CPF, datas de nascimento, CRM) por identificadores anônimos.

In [7]:
# Mapeamento nome/identificador -> anônimo (preenchido pelo pipeline)
MAPEAMENTO_ANONIMIZACAO = {}

# Chaveamento habilitação de anonimização
LIGAR_ANONIMIZACAO = True

def anonimizar_texto(texto):
    if LIGAR_ANONIMIZACAO:
        """Substitui CPF, CRM e nomes próprios por placeholders no texto."""
        if not isinstance(texto, str):
            return texto
        # CPF: xxx.xxx.xxx-xx -> [CPF_ANON]
        texto = re.sub(r"\d{3}\.\d{3}\.\d{3}-\d{2}", "[CPF_ANON]", texto)
        # CRM-XX 123456 -> [CRM_ANON]
        texto = re.sub(r"CRM-[A-Z]{2}\s*\d+", "[CRM_ANON]", texto)
        for nome_real, nome_anon in MAPEAMENTO_ANONIMIZACAO.items():
            texto = re.sub(re.escape(nome_real), nome_anon, texto, flags=re.IGNORECASE)
        return texto

def extrair_nomes_para_anonimizar(dados_dict):
    if LIGAR_ANONIMIZACAO:
        """Extrai nomes (paciente, medico) dos dados para criar mapeamento."""
        nomes = set()
        for nome_arq, conteudo in dados_dict.items():
            if isinstance(conteudo, list):
                for item in conteudo:
                    if isinstance(item, dict):
                        for k, v in item.items():
                            if v and isinstance(v, str) and k in ("paciente", "medico", "nome"):
                                nomes.add(v.strip())
            elif isinstance(conteudo, dict):
                for k, v in conteudo.items():
                    if k in ("paciente", "medico", "nome") and isinstance(v, str):
                        nomes.add(v.strip())
        return list(nomes)

def construir_mapeamento_anonimizacao(dados_dict):
    if LIGAR_ANONIMIZACAO:
        """Constrói mapa nome_real -> [TIPO_N] (ex: [PACIENTE_1], [MEDICO_1])."""
        global MAPEAMENTO_ANONIMIZACAO
        MAPEAMENTO_ANONIMIZACAO = {}
        nomes = extrair_nomes_para_anonimizar(dados_dict)
        for i, nome in enumerate(nomes, 1):
            MAPEAMENTO_ANONIMIZACAO[nome] = f"[PESSOA_{i}]"
        return MAPEAMENTO_ANONIMIZACAO

def anonimizar_objeto(obj):
    if LIGAR_ANONIMIZACAO:
        """Aplica anonimização recursiva em dict/list/str."""
        if isinstance(obj, str):
            return anonimizar_texto(obj)
        if isinstance(obj, list):
            return [anonimizar_objeto(x) for x in obj]
        if isinstance(obj, dict):
            return {k: anonimizar_objeto(v) for k, v in obj.items()}
        return obj

def executar_pipeline_anonimizacao(dados_brutos, salvar_em=None):
    if LIGAR_ANONIMIZACAO:
        """Pipeline completo: mapeamento + anonimização + opcionalmente salvar."""
        construir_mapeamento_anonimizacao(dados_brutos)
        dados_anon = anonimizar_objeto(dados_brutos)
        if salvar_em:
            Path(salvar_em).mkdir(parents=True, exist_ok=True)
            for nome, conteudo in dados_anon.items():
                path = Path(salvar_em) / f"{nome}.json"
                with open(path, "w", encoding="utf-8") as f:
                    json.dump(conteudo, f, ensure_ascii=False, indent=2)
        return dados_anon

# Executar após carregar dados
if 'dados_brutos' in dir() and dados_brutos:
    if LIGAR_ANONIMIZACAO:
        dados_anonimizados = executar_pipeline_anonimizacao(dados_brutos, DADOS_DIR)
        print("Anonimização concluída. Amostra:", list(dados_anonimizados.keys()))

Anonimização concluída. Amostra: ['dataset-pedidos-medicos', 'dataset-perguntas-respostas-medicos', 'dataset-pacientes', 'dataset-faq-medica', 'dataset-receitas', 'dataset-laudos', 'dataset-procedimentos', 'dataset-protocolos-medicos']


In [8]:
# Chaveamento habilitação de anonimização
# AVALIAR EXCLUSÃO DE BLOCO
LIGAR_ANONIMIZACAO = True

def dados_sinteticos_para_instrucoes(dados_anon):

    if LIGAR_ANONIMIZACAO:
        """Converte dados anonimizados em pares instrução/resposta para fine-tuning."""
        exemplos = []
        for nome_arq, conteudo in dados_anon.items():
            if not isinstance(conteudo, list):
                continue
            for item in conteudo:
                if isinstance(item, dict):
                    if "pergunta" in item and "resposta" in item:
                        exemplos.append({"instruction": item["pergunta"], "response": item["resposta"]})
                    elif "pergunta_medico" in item and "resposta" in item:
                        exemplos.append({"instruction": item["pergunta_medico"], "response": item["resposta"]})
                    elif "protocolo" in item and "descricao" in item:
                        inst = f"Qual o protocolo de {item.get('protocolo', '')}?"
                        exemplos.append({"instruction": inst, "response": item["descricao"]})
                    elif "exame" in item and "interpretacao" in item:
                        inst = f"Interpretar exame: {item.get('exame','')} - Resultado: {item.get('resultado','')}"
                        exemplos.append({"instruction": inst, "response": item["interpretacao"]})
                    elif "tipo" in item and "modelo" in item:
                        exemplos.append({"instruction": f"Modelo de {item['tipo']}", "response": item["modelo"]})
        return exemplos

def formatar_para_llama(instruction, response, template="default"):
    """Formata par instrução/resposta no estilo chat Llama."""
    if template == "default":
        text = f"<s>[INST] {instruction} [/INST] {response} </s>"
    else:
        text = f"Human: {instruction}\nAssistant: {response}"
    return text

def criar_dataset_finetuning(dados_anon, max_amostras=500):
    if LIGAR_ANONIMIZACAO:
        """Cria Dataset Hugging Face para treino."""
        exemplos = dados_sinteticos_para_instrucoes(dados_anon)[:max_amostras]
        if not exemplos:
            # fallback: criar exemplos genéricos a partir de FAQ/protocolos
            for nome_arq, conteudo in dados_anon.items():
                if isinstance(conteudo, list):
                    for item in conteudo:
                        if isinstance(item, dict):
                            for k, v in item.items():
                                if k in ("pergunta", "descricao", "resposta") and v:
                                    exemplos.append({"instruction": "Pergunta sobre saúde", "response": str(v)[:500]})
                    if len(exemplos) >= max_amostras:
                        break
        texts = [formatar_para_llama(e["instruction"], e["response"]) for e in exemplos]
        return Dataset.from_dict({"text": texts})

# CARGA DE MODELO E FINE-TUNING

## Etapa 3 – Carga do modelo base [TinyLlama/TinyLlama-1.1B-Chat-v1.0] do HuggingFace

Preparação dos dados no formato de instrução, fine-tuning com LangChain/Hugging Face (PEFT/LoRA) e salvamento do modelo customizado.

In [9]:
import os
from google.colab import userdata

# Modelo base open source Llama (TinyLlama para caber no Colab; pode trocar por meta-llama/Llama-2-7b-hf com mais RAM)
MODELO_BASE = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Chaveamento habilitação de anonimização
LIGAR_ANONIMIZACAO = True

def carregar_modelo_e_tokenizer(model_name=MODELO_BASE, use_4bit=True):
    """Carrega modelo e tokenizer para fine-tuning com quantização opcional."""
    hf_token = userdata.get('HF_TOKEN') # Pega o token armazenado nos Secrets do Google Colab

    bnb_config = BitsAndBytesConfig(load_in_4bit=use_4bit, bnb_4bit_compute_dtype="float16", bnb_4bit_quant_type="nf4") if use_4bit else None
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, token=hf_token)
    # Adiciona pad_token se não estiver definido para garantir que o tokenizador funcione corretamente com padding
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto", trust_remote_code=True, token=hf_token)
    if use_4bit:
        model = prepare_model_for_kbit_training(model)
    return model, tokenizer

def executar_finetuning(dados_anon, modelo_base=MODELO_BASE, output_dir=None, num_epochs=2, batch_size=2):
    if LIGAR_ANONIMIZACAO:
        """Executa fine-tuning (LoRA) e salva o modelo customizado."""
        output_dir = output_dir or str(MODELO_DIR)
        ds = criar_dataset_finetuning(dados_anon)
        model, tokenizer = carregar_modelo_e_tokenizer(modelo_base)

        lora_config = LoraConfig(r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none")
        model = get_peft_model(model, lora_config)

    def tokenize(examples):
        tokenized_inputs = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
        tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy() # Adiciona labels
        return tokenized_inputs

    ds_tokenized = ds.map(tokenize, batched=True, remove_columns=ds.column_names)

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        save_strategy="epoch",
        logging_steps=10,
        fp16=True,
    )
    trainer = Trainer(model=model, args=training_args, train_dataset=ds_tokenized)
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return output_dir

# Descomente para rodar (demora e exige GPU no Colab):
if 'dados_anonimizados' in dir():
    caminho_modelo = executar_finetuning(dados_anonimizados, output_dir=str(MODELO_DIR))
    print("Modelo salvo em:", caminho_modelo)


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,10.827344


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Modelo salvo em: /content/drive/MyDrive/tech_challenge_3/fine-tuning


# PREPARAÇÃO CHATGPT - Integração com o ChatGPT para análise de resultado dos exames

In [10]:
import os
import openai
from google.colab import userdata

def call_chatgpt_api(system_message: str, user_prompt_base: str, user_value: str) -> str:
    """
    Chama a API do ChatGPT com as mensagens de sistema e usuário fornecidas.

    Args:
        system_message (str): A mensagem de sistema para o modelo.
        user_prompt_base (str): A parte inicial da mensagem do usuário.
        user_value (str): O valor específico fornecido pelo usuário.

    Returns:
        str: A resposta do modelo ChatGPT-4, ou uma mensagem de erro.
    """
    chatgpt_api_key = userdata.get('FIAP-TechChallenger-Fase3-ChatGPT-API-KEY')

    if not chatgpt_api_key:
        return "Erro: A chave 'FIAP-TechChallenger-Fase3-ChatGPT-API-KEY' não foi encontrada nos secrets do Colab."

    client = openai.OpenAI(api_key=chatgpt_api_key)

    user_message_full = f"{user_prompt_base} {user_value}"

    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message_full}
            ],
            temperature=0.7,
            max_tokens=500
        )
        return response.choices[0].message.content

    except openai.APIError as e:
        return f"Erro da API do OpenAI: {e}"
    except Exception as e:
        return f"Ocorreu um erro inesperado ao chamar a API do ChatGPT: {e}"

print("Função 'call_chatgpt_api' criada com sucesso.")


Função 'call_chatgpt_api' criada com sucesso.


### Teste da função `call_chatgpt_api` de integração com o ChatGPT

In [11]:
# Exemplo de uso da função:
system_msg = "Como especialista em exames clínicos de um hospital, analise o resultado de exame informado:"
user_prompt = "O resultado do exame foi:"
user_val = "Cálculo renal de 5mm no rim direito"

print(f"Enviando prompt para o modelo GPT-4:\nSistema: {system_msg}\nUsuário: {user_prompt} {user_val}")
chatgpt_response = call_chatgpt_api(system_msg, user_prompt, user_val)

print("\n--- Resposta do ChatGPT-4 (via função) ---")
print(chatgpt_response)
print("------------------------------------------")


Enviando prompt para o modelo GPT-4:
Sistema: Como especialista em exames clínicos de um hospital, analise o resultado de exame informado:
Usuário: O resultado do exame foi: Cálculo renal de 5mm no rim direito

--- Resposta do ChatGPT-4 (via função) ---
O resultado do exame indica a presença de um cálculo renal, também conhecido como pedra nos rins, no rim direito do paciente. O cálculo tem 5mm, que é considerado um tamanho moderado.

Pedras deste tamanho podem, em alguns casos, ser passadas naturalmente através da urina, embora possam causar dor significativa durante o processo. No entanto, dependendo da dor e do desconforto do paciente, pode ser necessário tratamento médico, que pode incluir medicamentos para ajudar a passar a pedra, litotripsia extracorpórea por ondas de choque para quebrar a pedra, ou em casos mais graves, cirurgia.

É importante que o paciente beba muita água, evite alimentos ricos em oxalato (como espinafre, ruibarbo, nozes e trigo) e reduza a ingestão de sal e p

# VECTOR_STORE + RAG + LANGRAPH + ASSISTENTE

## Etapa 4 – Assistente com RAG

Construção do índice vetorial (RAG) e sistema de logs


In [12]:
# RAG, logging e geração de resposta

# RAG - preparação da funções

from typing import Optional # Import Optional for type hinting

def documentos_para_rag(dados_anon):
    """
    Converte dados anonimizados em uma lista de objetos Document para RAG.

    Args:
        dados_anon (dict): Dicionário com nome do arquivo e seu conteúdo.
    Returns:
        list: Lista de objetos Document.
    """
    docs = []

    for nome_arq, conteudo in dados_anon.items():
        # Caso o conteúdo seja uma lista de itens (ex: linhas de uma tabela)
        if isinstance(conteudo, list):
            for i, item in enumerate(conteudo):
                if isinstance(item, dict):
                    texto = json.dumps(item, ensure_ascii=False)
                    docs.append(Document(
                        page_content=texto,
                        metadata={"fonte": nome_arq, "indice": i}
                    ))

        # Caso o conteúdo seja um dicionário único
        elif isinstance(conteudo, dict):
            texto = json.dumps(conteudo, ensure_ascii=False)
            docs.append(Document(
                page_content=texto,
                metadata={"fonte": nome_arq}
            ))

    return docs

def construir_vectorstore(dados_anon, persist_path=None):

    """Cria FAISS a partir dos documentos e opcionalmente persiste."""
    docs = documentos_para_rag(dados_anon)
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
    splits = splitter.split_documents(docs)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    vectorstore = FAISS.from_documents(splits, embeddings)
    if persist_path:
       vectorstore.save_local(str(persist_path))
       return vectorstore

def log_operacao(operacao: str, detalhe: str, arquivo: str = LOG_OPERACOES) -> None:
    """
    Registra operação em logging-operacoes.csv com data e hora detalhadas.
    """
    from datetime import datetime
    import csv

    now = datetime.now()
    with open(arquivo, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if f.tell() == 0:
            w.writerow(["data", "hora", "minuto", "segundo", "operacao", "detalhe"])
        w.writerow([
            now.strftime("%Y-%m-%d"),
            now.strftime("%H"),
            now.strftime("%M"),
            now.strftime("%S"),
            operacao,
            detalhe,
        ])

# Logging enriquecido
def log_consulta(
    crm: str,
    pergunta: str,
    resposta_assistente: str,
    intencao: str,
    tipo_solicitacao: str,
    ramo_acionado: str,
    fonte_protocolo: str,
    grau_ctcae_max: Optional[str],
    arquivo: str = LOG_CONSULTAS,
) -> None:
    """
    Registra consulta em logging-consultas.csv com campos de auditoria clínica.
    """
    from datetime import datetime
    import csv

    now = datetime.now()
    with open(arquivo, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if f.tell() == 0:
            w.writerow([
                "data",
                "hora",
                "minuto",
                "segundo",
                "crm",
                "pergunta",
                "resposta_assistente",
                "intencao_detectada",
                "tipo_solicitacao",
                "ramo_acionado",
                "fonte_protocolo",
                "grau_ctcae_max",
            ])
        w.writerow([
            now.strftime("%Y-%m-%d"),
            now.strftime("%H"),
            now.strftime("%M"),
            now.strftime("%S"),
            crm,
            pergunta[:500],
            resposta_assistente[:1000],
            intencao,
            tipo_solicitacao,
            ramo_acionado,
            fonte_protocolo,
            grau_ctcae_max or "",
        ])

### Construção do VectorStore (FAISS)

In [13]:
# Construir o VectorStore para RAG
# Garante que dados_anonimizados (que agora são os dados brutos) estão sendo usados

print(f'VECTORSTORE_PATH: {VECTORSTORE_PATH}')

if 'dados_anonimizados' in dir() and dados_anonimizados:
    print("Construindo o VectorStore...")
    vectorstore = construir_vectorstore(dados_anonimizados, persist_path=VECTORSTORE_PATH)
    print(f"VectorStore construído e salvo em: {VECTORSTORE_PATH}")
else:
    print("Dados não disponíveis para construir o VectorStore. Certifique-se de que 'dados_brutos' foram carregados e 'dados_anonimizados' foram definidos.")

VECTORSTORE_PATH: /content/drive/MyDrive/tech_challenge_3/fine-tuning/faiss_index
Construindo o VectorStore...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

VectorStore construído e salvo em: /content/drive/MyDrive/tech_challenge_3/fine-tuning/faiss_index


## Etapa 5 – Fluxo de Execução com LangGraph

Roteamento de intenções (sugerir tratamentos, análise de exames, alertas de risco, dúvidas gerais) e assistente com logs e regras de segurança.

### Fluxo de Execução Langraph

[classificar] <br>→ analise_exames → [ramo_exames_gpt_ctcae] → [finalizar] → END
                      <br>→ demais          → [buscar_rag] → [ramo_rag_faiss] → [finalizar] → END

In [14]:
# LangGraph - versão mais atualizada da classe

import json
from collections import defaultdict
import re
from typing import TypedDict, Optional, List, Dict, Any

class EstadoAssistente(TypedDict, total=False):
    pergunta: str
    crm: str
    intencao: str                     # ex: "analise_exames", "tratamentos", "duvida_geral", "alertas_risco"
    tipo_solicitacao: str             # ex: "sintomas", "laudo", "protocolo", "geral"
    ramo_acionado: str                # ex: "ramo_exames_gpt_ctcae" ou "ramo_rag_faiss"
    contexto_rag: str
    resposta: str
    fonte: str                        # fonte principal dos protocolos usados
    grau_ctcae_max: Optional[str]     # G1..G5 quando houver exame
    metadados_rag: Optional[List[Dict[str, Any]]]  # opcional: lista com metadados dos documentos usados - quais documentos o FAISS retornou

# Classificação de intenção + tipo de solicitação
def classificar_intencao(pergunta: str) -> tuple[str, str]:
    """
    Retorna (intencao, tipo_solicitacao).

    intencao:
      - "analise_exames"
      - "tratamentos"
      - "alertas_risco"
      - "duvida_geral"

    tipo_solicitacao:
      - "laudo"
      - "sintomas"
      - "protocolo"
      - "geral"
    """
    texto = pergunta.lower()

    # Heurística simples para identificar que é laudo/exame
    if any(p in texto for p in ["hemograma", "exame", "laudo", "resultado do exame", "plaquetas", "hemoglobina"]):
        return "analise_exames", "laudo"

    if any(p in texto for p in ["dor", "náusea", "nausea", "febre", "dispneia", "falta de ar", "sintoma", "sintomas"]):
        return "tratamentos", "sintomas"

    if any(p in texto for p in ["protocolo", "conduta", "qual a dose", "dose de", "como tratar", "tratamento"]):
        return "duvida_geral", "protocolo"

    if any(p in texto for p in ["risco", "gravidade", "urgente", "emergência", "emergencia"]):
        return "alertas_risco", "geral"

    # fallback
    return "duvida_geral", "geral"

def node_classificar(state: EstadoAssistente) -> EstadoAssistente:
    intencao, tipo = classificar_intencao(state["pergunta"])
    state["intencao"] = intencao
    state["tipo_solicitacao"] = tipo
    return state

# RAG FAISS — guardar metadados e fonte
def node_buscar_rag(state: EstadoAssistente) -> EstadoAssistente:
    if "vectorstore" not in globals():
        raise RuntimeError("Vectorstore não inicializado. Execute a célula de construção do FAISS antes.")

    docs = vectorstore.similarity_search(state["pergunta"], k=10)
    contexto = "\n\n".join(d.page_content for d in docs)
    state["contexto_rag"] = contexto

    # guarda metadados para possível auditoria
    state["metadados_rag"] = [d.metadata for d in docs]

    # fonte principal (por ex. o arquivo mais recorrente)
    fontes = [d.metadata.get("fonte", "desconhecido") for d in docs]
    if fontes:
        # pega a fonte mais frequente
        from collections import Counter
        state["fonte"] = Counter(fontes).most_common(1)[0][0]
    else:
        state["fonte"] = "desconhecido"

    return state

def limpar_jsonl(input_string):
    linhas_limpas = []
    # Divide por quebra de linha
    linhas = input_string.strip().split('\n')

    for linha in linhas:
        # Remove espaços em branco nas extremidades
        linha_formatada = linha.strip()

        if not linha_formatada:
            continue

        try:
            # Valida se é um JSON válido
            objeto_json = json.loads(linha_formatada)
            # Reconverte para string em uma única linha (compacto)
            linhas_limpas.append(json.dumps(objeto_json, ensure_ascii=False))
        except json.JSONDecodeError as e:
            print(f"Erro ao processar linha: {linha_formatada[:30]}... Erro: {e}")

    return "\n".join(linhas_limpas)


### Ramo 1 — GPT‑4o + CTCAE

In [15]:
# classificar → ramo_exames_gpt_ctcae → finalizar → END.

import re

PADRAO_GRAU_CTCAE = re.compile(r"\bG([1-5])\b", flags=re.IGNORECASE)

def extrair_grau_ctcae(texto: str):
    """Extrai o maior grau CTCAE mencionado no texto (G1..G5), ou None."""
    graus = {int(m.group(1)) for m in PADRAO_GRAU_CTCAE.finditer(texto or "")}
    return f"G{max(graus)}" if graus else None

def node_ramo_exames_gpt_ctcae(state: EstadoAssistente) -> EstadoAssistente:
    """
    Ramo 1: envia o laudo/exame ao GPT-4 para análise clínica
    e classificação segundo a escala CTCAE (G1–G5).
    """
    systemmsg = (
        "Você é um médico especialista em exames laboratoriais de um hospital. "
        "Analise os exames informados, descreva achados relevantes e atribua um grau de toxicidade "
        "segundo a escala CTCAE (Critérios de Terminologia Comum para Eventos Adversos), de G1 a G5. "
        "Apresente os resultados de forma estruturada e cite explicitamente os graus, "
        "por exemplo: 'Neutropenia: G3', 'Anemia: G2'."
    )

    resposta_exames = call_chatgpt_api(
        system_message=systemmsg,
        user_prompt_base="Resultados de exame do paciente: ",
        user_value=state["pergunta"]
    )

    state["resposta"]       = resposta_exames
    state["ramo_acionado"]  = "ramo_exames_gpt_ctcae"
    state["grau_ctcae_max"] = extrair_grau_ctcae(resposta_exames)
    return state


### Ramo 2 — RAG FAISS + TinyLlama

In [16]:
# classificar → buscar_rag → ramo_rag_faiss → finalizar → END;
def gerar_resposta_tinyllama(pergunta: str, contexto: str) -> str:
    """
    Usa o modelo fine-tunado TinyLlama + contexto RAG para gerar resposta.
    Adapte esta função para chamar seu pipeline atual (modelo + tokenizer).
    """
    prompt = (
        "Você é um assistente médico de emergência que responde com base em protocolos internos do hospital. "
        "Use apenas as informações do contexto abaixo e responda de forma objetiva, "
        "sempre reforçando que o médico é o responsável final pela decisão.\n\n"
        "=== CONTEXTO ===\n"
        f"{contexto}\n\n"
        "=== PERGUNTA DO MÉDICO ===\n"
        f"{pergunta}\n\n"
        "=== RESPOSTA ===\n"
    )

    # Aqui você pluga sua função de geração com TinyLlama fine-tunado
    # Exemplo esquemático (adapte para o seu código):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.3,
        top_p=0.9
    )
    resposta = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Se necessário, pós-processar para remover o prompt inicial
    return resposta[len(prompt):].strip()


def node_ramo_rag_faiss(state: EstadoAssistente) -> EstadoAssistente:
    contexto = state.get("contexto_rag", "")
    pergunta = state.get("pergunta", "")

    prompt = (
        "Você é um assistente médico de emergência que responde com base nos "
        "protocolos internos do hospital. Use apenas as informações do contexto abaixo "
        "e responda de forma objetiva, sempre reforando que o médico é o responsável "
        "final pela decisão clínica.\n"
        f"CONTEXTO:\n{contexto}\n\n"
        f"PERGUNTA DO MÉDICO:\n{pergunta}"
        f"{MARCADOR_RESPOSTA}"   # ← mesmo marcador que inferencia_llm_simulada usa para cortar
    )

    resposta = inferencia_llm_simulada(prompt, modelopath=str(MODELO_DIR))

    state["resposta"]       = resposta
    state["ramo_acionado"]  = "ramo_rag_faiss"
    return state

In [17]:
# ============================================================
# GRAFO LANGGRAPH — RAMOS EXPLÍCITOS + TRAVA ÉTICA + INFERÊNCIA
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


# ----------------------------------------------------------
# Trava ética reforçada (patch 6)
# ----------------------------------------------------------
def aplicar_regras_seguranca(texto: str) -> str:
    """
    Aplica trava ética:
    - Evita prescrição de medicamentos
    - Atenua diagnósticos categóricos
    - Reforça supervisão médica obrigatória
    """
    if not isinstance(texto, str):
        return texto

    out = texto

    # Bloqueio de prescrições / medicamentos
    if any(p in out.lower() for p in ["medicamento", "remédio", "remedio", "dose de ", "mg ", "posologia"]):
        out += (
            "\n\nObservação importante: não sugiro medicamentos. "
            "A escolha de fármacos, dosagem e posologia deve ser feita exclusivamente "
            "pelo médico responsável pelo caso."
        )

    # Atenuar diagnósticos muito categóricos
    padroes_diag = [
        "você tem ",
        "voce tem ",
        "o paciente tem ",
        "diagnóstico de ",
        "diagnostico de "
    ]
    if any(p in out.lower() for p in padroes_diag):
        out += (
            "\n\nAtenção: qualquer hipótese diagnóstica mencionada deve ser "
            "confirmada pelo médico assistente, com base em avaliação clínica completa "
            "e exames complementares."
        )

    # Reforço de supervisão médica sempre
    if "confirme com seu médico" not in out.lower():
        out += (
            "\n\nRecomendo sempre confirmar estas informações com o médico que acompanha o caso "
            "ou o médico de plantão."
        )

    return out


# ----------------------------------------------------------
# Inferência LLM (TinyLlama fine-tunado) — mantida do original
# ----------------------------------------------------------

# Marcador fixo — usado na montagem do prompt E no corte da resposta
MARCADOR_RESPOSTA = "\nRESPOSTA:\n"

def inferencia_llm_simulada(prompt: str, modelopath=None) -> str:
    """Inferência com modelo fine-tunado ou simulada se modelo não disponível."""
    print(f" modelopath considerado para inferência: {modelopath}")

    try:
        base_model, tokenizer = carregar_modelo_e_tokenizer(model_name=MODELO_BASE)
        model = base_model

        if modelopath and Path(modelopath).exists():
            print(f"DEBUG: Tentando carregar adaptador PEFT de {modelopath}")
            try:
                model = PeftModel.from_pretrained(base_model, modelopath)
                model = model.merge_and_unload()
                print("DEBUG: Adaptador PEFT carregado e mesclado com sucesso.")
            except Exception as peft_e:
                print(f"DEBUG: Erro ao carregar/mesclar adaptador PEFT de {modelopath}: {peft_e}")
                print("DEBUG: Caindo para inferência com o modelo base sem fine-tuning.")
                model = base_model
        else:
            print(f"DEBUG: Caminho do modelo fine-tuned '{modelopath}' não fornecido ou não existe. Usando modelo base.")
            model = base_model

        if not model or not tokenizer:
            return "Falha crítica ao carregar modelo LLM. Sempre consulte seu médico para condutas."

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

        texto_completo = tokenizer.decode(out[0], skip_special_tokens=True)

        # ── CORTE PELO MARCADOR FIXO ──────────────────────────────────────────
        # O prompt termina com MARCADOR_RESPOSTA; pegamos só o que vem depois
        if MARCADOR_RESPOSTA in texto_completo:
            partes = texto_completo.split(MARCADOR_RESPOSTA)
            resposta = partes[-1].strip()       # última parte = geração do modelo
        else:
            # fallback: remove pelo tamanho do prompt tokenizado
            n_tokens_prompt = inputs["input_ids"].shape[1]
            resposta_ids = out[0][n_tokens_prompt:]
            resposta = tokenizer.decode(resposta_ids, skip_special_tokens=True).strip()

        # Garante que não vaza o bloco "PERGUNTA DO MÉDICO" para a resposta
        for corte in ["PERGUNTA DO MÉDICO", "=== PERGUNTA", "### PERGUNTA"]:
            if corte in resposta:
                resposta = resposta.split(corte)[0].strip()

        if not resposta:
            resposta = "Não foi possível gerar uma resposta baseada nos protocolos disponíveis."

        return resposta

    except Exception as e:
        print(f"DEBUG: Erro geral durante a inferência do LLM: {e}")
        log_operacao("erro_inferencia", str(e))
        return f"Erro na inferência do LLM: {e}. Sempre consulte seu médico para condutas."

# ----------------------------------------------------------
# Nó terminal — aplica trava ética em toda resposta (patch 7)
# ----------------------------------------------------------
def node_finalizar(state: EstadoAssistente) -> EstadoAssistente:
    """Nó terminal: aplica regras de segurança/ética antes de entregar ao médico."""
    state["resposta"] = aplicar_regras_seguranca(state.get("resposta", ""))
    return state


# ----------------------------------------------------------
# Função de roteamento condicional (patch 7)
# ----------------------------------------------------------
def roteamento(state: EstadoAssistente) -> str:
    """
    Após classificar intenção, decide o ramo:
      analise_exames → Ramo 1 (GPT-4o / CTCAE)
      demais         → Ramo 2 (buscar_rag → TinyLlama)
    """
    if state.get("intencao") == "analise_exames":
        return "ramo_exames_gpt_ctcae"
    return "buscar_rag"


# ----------------------------------------------------------
# Montagem do grafo LangGraph com ramos explícitos (patch 7)
# Substitui a função construir_grafo_assistente() original
# ----------------------------------------------------------

#                      ┌──► ramo_exames_gpt_ctcae ──────────────┐
#classificar ──►[cond] │                                        ├──► finalizar ──► END
#                      └──► buscar_rag ──► ramo_rag_faiss ──────┘

def construir_grafo_assistente(vectorstore, llm_inference_fn=None):
    """
    Monta o grafo LangGraph com dois ramos explícitos:

    Ramo 1 (analise_exames):
        classificar → ramo_exames_gpt_ctcae → finalizar → END

    Ramo 2 (tratamentos / duvida_geral / alertas_risco):
        classificar → buscar_rag → ramo_rag_faiss → finalizar → END
    """

    # Encapsula node_buscar_rag com o vectorstore injetado
    def buscar(state):
        return node_buscar_rag(state)

        # return node_buscar_rag(state, vectorstore)

    graph = StateGraph(EstadoAssistente)

    # Registro dos nós
    graph.add_node("classificar",           node_classificar)
    graph.add_node("buscar_rag",            buscar)
    graph.add_node("ramo_exames_gpt_ctcae", node_ramo_exames_gpt_ctcae)  # Ramo 1
    graph.add_node("ramo_rag_faiss",        node_ramo_rag_faiss)         # Ramo 2
    graph.add_node("finalizar",             node_finalizar)

    # Ponto de entrada
    graph.set_entry_point("classificar")

    # Edge condicional após classificação
    graph.add_conditional_edges(
        "classificar",
        roteamento,
        {
            "ramo_exames_gpt_ctcae": "ramo_exames_gpt_ctcae",  # Ramo 1
            "buscar_rag":            "buscar_rag",              # Ramo 2
        }
    )

    # Ramo 1: GPT-4o/CTCAE → finalizar → END
    graph.add_edge("ramo_exames_gpt_ctcae", "finalizar")

    # Ramo 2: buscar_rag → ramo_rag_faiss → finalizar → END
    graph.add_edge("buscar_rag",    "ramo_rag_faiss")
    graph.add_edge("ramo_rag_faiss", "finalizar")

    # Saída comum
    graph.add_edge("finalizar", END)

    return graph.compile()


# ----------------------------------------------------------
# Compilação do workflow (uso direto)
# ----------------------------------------------------------
# Certifique-se de que 'vectorstore' já foi criado na célula anterior
if "vectorstore" in dir():
    workflow = construir_grafo_assistente(vectorstore)
    print("✅ Grafo LangGraph compilado com sucesso.")
    print("   Ramo 1 (analise_exames) : classificar → ramo_exames_gpt_ctcae → finalizar → END")
    print("   Ramo 2 (demais)         : classificar → buscar_rag → ramo_rag_faiss → finalizar → END")
else:
    print("⚠️  Vectorstore não encontrado. Execute a célula de construção do FAISS antes desta.")

✅ Grafo LangGraph compilado com sucesso.
   Ramo 1 (analise_exames) : classificar → ramo_exames_gpt_ctcae → finalizar → END
   Ramo 2 (demais)         : classificar → buscar_rag → ramo_rag_faiss → finalizar → END


# TESTE DO ASSISTENTE


## Etapa 6 – Executa uma consulta através do fluxo do LangGraph



### Testando o modelo fine-tuned e do Assistente

In [18]:
# Assistente Pergunta

def executar_assistente_pergunta(pergunta: str, crm: str, grafo=None) -> str:
    grafo_usar = grafo or workflow          # fallback para compatibilidade

    estado_inicial = {                      # ← sempre local, nunca global
        "pergunta": pergunta,
        "crm":      crm,
    }

    state_final = grafo_usar.invoke(estado_inicial)
    resposta_final = state_final.get("resposta", "")

    log_consulta(
        crm               = crm,
        pergunta          = pergunta,
        resposta_assistente = resposta_final,
        intencao          = state_final.get("intencao",         ""),
        tipo_solicitacao  = state_final.get("tipo_solicitacao", ""),
        ramo_acionado     = state_final.get("ramo_acionado",    ""),
        fonte_protocolo   = state_final.get("fonte",            ""),
        grau_ctcae_max    = state_final.get("grau_ctcae_max"),
    )
    return resposta_final

In [19]:
# Teste para o assistente com RAG e, opcionalmente, com o modelo fine-tuned.

if "dados_anonimizados" in dir() and dados_anonimizados:
    print("Preparando para testar o assistente...")

    # Reconstruir o vectorstore para RAG
    # Isso garante que o vectorstore esteja carregado ou atualizado para o teste
    vectorstore_test = construir_vectorstore(dados_anonimizados, persist_path=VECTORSTORE_PATH)

    # Definir se usará o modelo fine-tunado (True/False) para este teste
    # Alterado para True para testar o comportamento do LLM e a busca de todos os resultados RAG
    usar_modelo_finetuned_test = True

    llm_fn_test = (lambda p: inferencia_llm_simulada(p, str(MODELO_DIR))) if usar_modelo_finetuned_test else None

    # Construir o grafo do assistente para o teste
    grafo_test = construir_grafo_assistente(vectorstore_test, llm_fn_test)

    # Simular uma consulta
    pergunta_teste = "Paciente com náusea persistente e perda de apetite."
    crm_teste = "123456"

    print(f"\nConsultando o assistente (crm: {crm_teste}, assunto: \"{pergunta_teste}\")...")
    resposta_teste = executar_assistente_pergunta(pergunta_teste, crm_teste, grafo_test)

    print("\n\n\n\n\n--- Resposta do Assistente Médico ---")
    print(resposta_teste)
    print("-------------------------------------")
    print("\n Teste do assistente concluído.")

else:
    print("Dados anonimizados não disponíveis. Certifique-se de que os dados sintéticos foram carregados e o pipeline de anonimização (mesmo que bypassado) foi executado.")

Preparando para testar o assistente...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Consultando o assistente (crm: 123456, assunto: "Paciente com náusea persistente e perda de apetite.")...
 modelopath considerado para inferência: /content/drive/MyDrive/tech_challenge_3/fine-tuning


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

DEBUG: Tentando carregar adaptador PEFT de /content/drive/MyDrive/tech_challenge_3/fine-tuning


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


DEBUG: Adaptador PEFT carregado e mesclado com sucesso.





--- Resposta do Assistente Médico ---
Insuficiência renal crônica é a perda progressiva e geralmente irreversível da função dos rins. O tratamento visa retardar a progressão (controle de pressão, glicemia, dieta), tratar complicações e, em estágios avançados, pode ser indicada diálise ou transplante. O nefrologista define o plano de cuidado.

Recomendo sempre confirmar estas informações com o médico que acompanha o caso ou o médico de plantão.
-------------------------------------

 Teste do assistente concluído.


# PROGRAMA PRINCIPAL

### . Garante que os dados anonimizados sejam carregados e reconstrói a base vetorial.
### . Define se o modelo ajustado deve ser utilizado.
### . Constrói o grafo (fluxo de trabalho) do assistente.
### . Em seguida, entra em um loop infinito, solicitando ao usuário o seu CRM e as suas perguntas.
### . Processa cada pergunta utilizando a função executar_assistente_pergunta,exibe a resposta e registra os detalhes da sessão.
### . O loop é interrompido se o usuário digitar 'sair'.

In [ ]:
from IPython.display import clear_output

## PROGRAMA PRINCIPAL DO ASSISTENTE MÉDICO

clear_output(wait=True)
print("Inicializando o assistente médico interativo...")

# Certifique-se de que 'dados_anonimizados' está disponível do pipeline anterior
if "dados_anonimizados" not in dir() or not dados_anonimizados:
    print("Nenhum dado anonimizado carregado. Carregue os dados sintéticos e execute a anonimização antes de rodar o assistente interativo.")
else:
    # Construir o vectorstore para RAG
    # Reconstruímos aqui para garantir que o vectorstore esteja atualizado ou carregado
    vectorstore = construir_vectorstore(dados_anonimizados, persist_path=VECTORSTORE_PATH)

    # Definir se usará o modelo fine-tunado (True/False)
    # Por padrão, manteremos como False, a menos que o fine-tuning tenha sido explicitamente executado.
    usar_modelo_finetuned = True # Altere para True se você executou o fine-tuning
    llm_fn = (lambda p: inferencia_llm_simulada(p, str(MODELO_DIR))) if usar_modelo_finetuned else None

    # Construir o grafo do assistente com o vectorstore e a função de inferência do LLM
    grafo = construir_grafo_assistente(vectorstore, llm_fn)

    print("\n\n Assistente pronto. Digite 'sair' a qualquer momento para encerrar.\n")

    crm = input("\n Informe seu CRM:\n")

    log_operacao("inicio_sessao", f"{crm}")

    while True:
        pergunta = input("\n Sintomas ou exames:  (digite sair para encerrar) \n").strip().lower()
        if not pergunta or pergunta.lower() == 'sair':
            print("Encerrando o assistente.")
            break

        print("\n\n Processando sua pesquisa...\n\n ")
        resposta = executar_assistente_pergunta(pergunta, crm, grafo)

        print("\n\n--- Assistente Médico --- \n")
        print(resposta)
        print("------------------------- \n\n")

    log_operacao("fim_sessao", crm)
    print("Sessão encerrada. Logs em logging-operações.csv e logging-consultas.csv.")

Inicializando o assistente médico interativo...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.




 Assistente pronto. Digite 'sair' a qualquer momento para encerrar.


 Informe seu CRM:
3452345234

 Sintomas ou exames:  (digite sair para encerrar) 
paciente com síndrome de Shulman e hipertrófia renal.


 Processando sua pesquisa...

 
 modelopath considerado para inferência: /content/drive/MyDrive/tech_challenge_3/fine-tuning


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

DEBUG: Tentando carregar adaptador PEFT de /content/drive/MyDrive/tech_challenge_3/fine-tuning


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


DEBUG: Adaptador PEFT carregado e mesclado com sucesso.


--- Assistente Médico --- 

Avaliação de causa pré-renal, renal e pós-renal; correção de hipovolemia; evitar nefrotóxicos; ajuste de doses de medicamentos; indicação de diálise conforme critérios clínicos e laboratoriais (K+, acidose, overload, uremia).
INDICACOES:
Elevação de creatinina ou queda de diurese em relação à baseline.
INDICACOES:
O que é insuficiência renal crônica?
Sugere-se classificar como lesão renal aguda ou ag

Observação importante: não sugiro medicamentos. A escolha de fármacos, dosagem e posologia deve ser feita exclusivamente pelo médico responsável pelo caso.

Recomendo sempre confirmar estas informações com o médico que acompanha o caso ou o médico de plantão.
------------------------- 



 Sintomas ou exames:  (digite sair para encerrar) 
RESONÂNCIA MAGNÉTICA DO JOELHO DIREITO Indicação clínica: Entorse Técnica: Foram obtidas imagens multiplanares nas sequências pesadas em T1, T2 e em densidade protônica

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

DEBUG: Tentando carregar adaptador PEFT de /content/drive/MyDrive/tech_challenge_3/fine-tuning
DEBUG: Adaptador PEFT carregado e mesclado com sucesso.


--- Assistente Médico --- 

a imagem multiplanar nas sequências pesadas em t1, t2 e em densidade protônica mostrou a presença de um deslocamento medialmente do segmento menisco do lado esquerdo, com fragmento meniscal deslocado, indicando a presença de uma lesão meniscal tipo alça de balde.
Avaliando a conduta, a lesão foi classificada como lesão renal aguda ou agudização sobre crônica. Avaliando a triagem rápida (NIHSS), a TC crânio sem contraste, a glicemia e pressão arterial controladas, a aval

Recomendo sempre confirmar estas informações com o médico que acompanha o caso ou o médico de plantão.
------------------------- 


